# 07 — Correct β schedule & Moreau envelope tracking

Two changes from the paper (Davis & Drusvyatskiy, 2019):

1. **Constant β_t = β** — Algorithm 4.1 uses a nondecreasing sequence {β_t} ⊆ (ρ̄, ∞). The paper's experiments (Section 5) use a constant β, sweeping over 100 values of β⁻¹ ∈ [10⁻⁴, 1]. We replace the old growing schedule with a constant β.

2. **Moreau envelope gradient norm** — The paper's convergence measure is ‖∇φ_λ(x)‖ where φ_λ is the Moreau envelope with λ = 1/(2ρ). This is computed as (1/λ)‖x - prox_{λφ}(x)‖. We add this as a tracking metric.

In [ ]:
import torch
import matplotlib.pyplot as plt

from src.PhaseRetrievalProblem import PhaseRetrievalProblem
from src.ModelBasedSolver import ModelBasedSolver

## Phase retrieval setup (same as Section 5.1 of the paper)

In [ ]:
torch.manual_seed(42)
d = 10
m = 100
true_x = torch.randn(d)
true_x = true_x / torch.norm(true_x)  # unit sphere, as in the paper
a_matrix = torch.randn(m, d)
b_vector = (a_matrix @ true_x) ** 2

population_data = torch.cat([a_matrix, b_vector.unsqueeze(1)], dim=1)
print(f"d={d}, m={m}, ||true_x||={torch.norm(true_x).item():.4f}")

## Test constant β schedule

Algorithm 4.1 requires β > ρ̄ > τ+η. For the prox-linear method on phase retrieval, τ=0 and η=ρ, so we need β > ρ. Default is β = 2ρ+1.

In [ ]:
prob = PhaseRetrievalProblem(rho=2.0)
x_init = torch.randn(d)
x_init = x_init / torch.norm(x_init)  # unit sphere

solver = ModelBasedSolver(
    problem=prob,
    data=population_data,
    x_init=x_init,
    T=500,
    batch_size=1,
    beta=5.0,       # constant β > ρ = 2
    log_every=50,
    moreau_every=100,
)

final_x = solver.run()
print(f"\nReconstruction error: {torch.norm(final_x - true_x).item():.4f}")

## Plot convergence: objective value & Moreau gradient norm

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Objective value
ax1.plot(solver.history["iterations"], solver.history["obj_values"])
ax1.set_xlabel("Iteration")
ax1.set_ylabel(r"$\varphi(x_t)$")
ax1.set_title("Population objective")
ax1.set_yscale("log")
ax1.grid(True, alpha=0.3)

# Moreau envelope gradient norm
if solver.history["moreau_grad_norms"]:
    iters_m, norms_m = zip(*solver.history["moreau_grad_norms"])
    ax2.plot(iters_m, norms_m, marker="o")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel(r"$\|\nabla \varphi_\lambda(x_t)\|$")
    ax2.set_title(r"Moreau envelope gradient norm ($\lambda = 1/2\rho$)")
    ax2.set_yscale("log")
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Compare different β values

The paper sweeps 100 values of β⁻¹ ∈ [10⁻⁴, 1]. Here we test a few to see sensitivity.

In [ ]:
beta_values = [3.0, 5.0, 10.0, 50.0, 100.0]
results = {}

for beta in beta_values:
    torch.manual_seed(42)
    x_init = torch.randn(d)
    x_init = x_init / torch.norm(x_init)

    solver = ModelBasedSolver(
        problem=prob,
        data=population_data,
        x_init=x_init,
        T=500,
        batch_size=1,
        beta=beta,
        log_every=10,
    )
    solver.run()
    results[beta] = solver.history
    print(f"β={beta:6.1f} | final obj = {solver.history['obj_values'][-1]:.6f}")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for beta, hist in results.items():
    ax.plot(hist["iterations"], hist["obj_values"], label=f"β={beta}")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"$\varphi(x_t)$")
ax.set_title("Convergence for different β (phase retrieval, prox-linear)")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()